# Bounded Self-Instruct → Qwen3-8B SFT → Lighteval loop

GPT-6 Luna is the parent model, accessed through the OpenAI Responses API. GPU workers load quantized student replicas with the current LoRA adapter for each phase. The notebook coordinates API calls and holds no model weights. Smoke mode generates teacher-checked synthetic pairs and evaluates with native Lighteval metrics. Full mode uses a mixed teacher/template curriculum and a bundled fixed synthetic evaluation set, with regression checks and no manual review or API evaluation-data generation. Independently reviewed evaluation remains an optional separate mode.

Student predictions are diagnostics. The student's forward pass in SFTTrainer is trained against parent completion tokens; incorrect sampled student responses are never used as labels. This notebook does not claim independent medical validation. Run cells in order on a CUDA GPU with Internet enabled.

Kaggle T4 x2 mode: restart the kernel, expose GPUs 0 and 1, and keep the notebook free of model weights. Separate workers use DDP for training and batched parallel inference. Existing adapter checkpoints and completed data remain reusable; floating-point results may differ across hardware layouts.


In [ ]:
from pathlib import Path
import os
import subprocess

# Configure CUDA before any torch import. Restart first if CUDA was already used.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Fresh Kaggle sessions need the complete repository before project discovery.
# Outside Kaggle, open this notebook from your existing complete checkout.
KAGGLE_REPO = Path("/kaggle/working/Health-Insurance-SLM")
if Path("/kaggle/working").is_dir():
    if not (Path.cwd() / "requirements-training.txt").is_file() and not KAGGLE_REPO.exists():
        subprocess.run([
            "git", "clone", "--branch", "Distillation",
            "https://github.com/j6antony/Health-Insurance-SLM.git", str(KAGGLE_REPO),
        ], check=True)
    elif KAGGLE_REPO.exists():
        print("Reusing existing checkout:", KAGGLE_REPO)
    print("Cloning does not include unpushed local changes. Existing checkouts are not automatically updated.")


In [ ]:
from pathlib import Path
import os
import sys

# Two independent GPU workers: set BEFORE importing torch.
# Restart the kernel first if CUDA was initialized in an earlier cell.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Attach/clone the complete repository, not just this notebook.
# Override PROJECT_ROOT explicitly if your checkout is elsewhere.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "requirements-training.txt").is_file():
    candidates = []
    for parent in (Path("/kaggle/working"), Path("/kaggle/input")):
        if parent.exists():
            candidates.extend(parent.rglob("requirements-training.txt"))
    candidates = [p.parent for p in candidates if (p.parent / "run_self_instruct.py").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set PROJECT_ROOT to the complete repository checkout, then rerun this cell.")
    PROJECT_ROOT = candidates[0]
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-training.txt")
print("Project:", PROJECT_ROOT)

# Catch an old or incomplete clone before imports, downloads, or paid calls.
required_files = [
    "distillation_loop/preflight.py", "distillation_loop/review.py",
    "distillation_loop/quality.py", "distillation_loop/trusted_templates.py",
    "self_instruct_pipeline/rule_checks.py",
]
missing_files = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing_files:
    raise RuntimeError(
        "Repository is outdated or incomplete: " + ", ".join(missing_files)
        + ". Upload/sync the complete updated repository, restart the kernel, and rerun from the top."
    )


In [ ]:
%pip install -q -r "{REQUIREMENTS}"
# If an earlier cell imported incompatible library versions, restart the kernel
# once after installation, then rerun from the top.


After dependency installation, restart the kernel if required and rerun setup cells from the top. Do not insert an automatic kernel-exit cell into Run All. Enable Kaggle T4 ×2 and Internet before continuing.


In [ ]:
import os

# Only the OpenAI parent needs a credential. Qwen3-8B loads anonymously below.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY").strip()
    except Exception as exc:
        raise RuntimeError("Add OPENAI_API_KEY to Kaggle Secrets and enable notebook access.") from None
if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise RuntimeError("OPENAI_API_KEY is empty.")
print("OpenAI credential loaded. Hugging Face login is not required.")

# Optional durable cost history. First run supabase/api_usage.sql in Supabase.
# Add BOTH values in Kaggle Secrets and enable notebook access to each.
for name in ("SUPABASE_URL", "SUPABASE_SECRET_KEY"):
    if not os.environ.get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ[name] = UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
from self_instruct_pipeline.usage_cloud import SupabaseLedger
cloud_ledger = SupabaseLedger.from_environment()
if cloud_ledger:
    cloud_ledger.check()
    print("Supabase cost ledger connected. Calls will sync automatically.")
else:
    print("Cost logging is LOCAL ONLY. Add Supabase secrets for history across sessions.")

# Optional private dataset uploads (disabled; use the ZIP download cell below).
# Add KAGGLE_USERNAME (your account name) and either KAGGLE_API_TOKEN or KAGGLE_KEY.
for name in ("KAGGLE_USERNAME", "KAGGLE_API_TOKEN", "KAGGLE_KEY"):
    if not os.environ.get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ[name] = UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
BACKUP_ADAPTERS_TO_KAGGLE = False
if BACKUP_ADAPTERS_TO_KAGGLE:
    owner = os.environ.get("KAGGLE_USERNAME", "").strip()
    if not owner:
        raise RuntimeError("Add your account name as KAGGLE_USERNAME in Kaggle Secrets for adapter backups.")
    os.environ["KAGGLE_ADAPTER_DATASET"] = owner + "/health-insurance-slm-adapters"
    from distillation_loop.kaggle_backup import check_backup_setup
    check_backup_setup()
    print("Private Kaggle adapter backup enabled.")
else:
    os.environ.pop("KAGGLE_ADAPTER_DATASET", None)

# GitHub is the default remote destination for completed adapter checkpoints.
UPLOAD_ADAPTERS_TO_GITHUB = True
os.environ["GITHUB_ADAPTER_UPLOAD"] = "1" if UPLOAD_ADAPTERS_TO_GITHUB else "0"
if UPLOAD_ADAPTERS_TO_GITHUB and not os.environ.get("GITHUB_TOKEN", "").strip():
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    except Exception:
        raise RuntimeError("Add and enable GITHUB_TOKEN in Kaggle Secrets, scoped to j6antony/Health-Insurance-SLM with Contents read/write.") from None
if UPLOAD_ADAPTERS_TO_GITHUB and not os.environ.get("GITHUB_TOKEN", "").strip():
    raise RuntimeError("GITHUB_TOKEN is empty.")
print("GitHub adapter uploads enabled." if UPLOAD_ADAPTERS_TO_GITHUB else "Local ZIP export only.")


In [ ]:
# Restart the kernel before switching from the old single-GPU notebook.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import torch
from transformers import AutoTokenizer
from distillation_loop.evaluation import require_training_versions
require_training_versions()
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Enable Kaggle T4 x2 and restart the kernel."
MODEL_NAME = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
student = None  # Worker processes own model weights; keep notebook GPU memory free.
print("Visible GPUs:", torch.cuda.device_count())
print("GPU workers:", [torch.cuda.get_device_name(i) for i in range(2)])


## Loop settings

**Full runs (`TEST_RUN = False`) split the bundled synthetic benchmark into 432 development cases and 108 final holdout cases by source family.** The version, hash, schemas, evidence and coverage are checked locally. No parent API calls generate these 540 cases, and no manual review is required. Missing or changed evaluation files stop with an error instead of switching to paid generation.

The student answers the same fixed cases before training and after each round. Those answers still require GPU inference. Each round targets 72 new training examples (54 teacher-generated and 18 code-generated rule examples with the default 25% mixture), with cumulative replay. API credits still pay for teacher training-data generation and verification.

The evaluation set is locally authored synthetic data with 10 case families and 3 challenge variants per task, not independent clinical gold. Set `TEST_RUN = True` for the smaller smoke test, which also selects 18 cases from the bundled corpus without API evaluation-data generation. Full runs use a separate `distillation_loop_adaptive_v1` folder; earlier run folders remain intact. GitHub adapter uploads remain enabled. The API budget remains enforced.


In [ ]:
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import atomic_json
from distillation_loop.core import load_loop_config
from distillation_loop.dual_gpu import DualGPURuntime

# False runs full training with the bundled evaluation set. True selects a small smoke test.
TEST_RUN = False

parent_config = load_config(PROJECT_ROOT / "config/self_instruct.json")
loop_config = load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json"))

# Full run: load the bundled fixed evaluation set. No API evaluation-data generation.
loop_config["evaluation"]["mode"] = "fixed"

# Editable before starting a new run:
loop_config["max_rounds"] = 5
loop_config["thresholds"] = {
    "macro_task_score": 0.90,
    "minimum_task_score": 0.80,
    "schema_evidence_valid_rate": 0.98,
}
RUN_DIR = (Path("/kaggle/working/distillation_loop_adaptive_v1") if Path("/kaggle/working").exists()
           else PROJECT_ROOT / "outputs/distillation_loop_adaptive_v1")
if TEST_RUN:
    from distillation_loop.smoke import smoke_configs
    RUN_DIR = RUN_DIR.with_name("distillation_smoke_v2")
    parent_config, loop_config = smoke_configs(parent_config, loop_config, RUN_DIR)
print("SMOKE TEST" if TEST_RUN else "FULL RUN", "Output:", RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if cloud_ledger:
    from self_instruct_pipeline.usage_cloud import sync_usage_history
    print(sync_usage_history(RUN_DIR))  # Import prior local costs before new paid calls.
loop_config["output_dir"] = str(RUN_DIR)
# Validate notebook overrides using the same configuration loader.
atomic_json(RUN_DIR / "requested_loop_config.json", loop_config)
loop_config = load_loop_config(str(RUN_DIR / "requested_loop_config.json"))
# Read-only data/configuration check before model workers or paid calls.
from distillation_loop.preflight import readiness
readiness_report = readiness(parent_config, loop_config, check_runtime=False)
print("Readiness:", readiness_report)
if not readiness_report["ready"]:
    raise RuntimeError("Resolve readiness issues above before running the experiment.")
student_runtime = DualGPURuntime(tokenizer, loop_config, MODEL_NAME, batch_size=2)
student_runtime.prepare()  # Checks two GPUs; workers load/train their own LoRA models.
print("Max rounds:", loop_config["max_rounds"])
print("Thresholds:", loop_config["thresholds"])


In [ ]:
# Optional request preview: this cell makes no parent API calls.
import copy
from self_instruct_pipeline.pipeline import Pipeline
preview_config = copy.deepcopy(parent_config)
preview_config["paths"]["output_dir"] = str(RUN_DIR / "preview")
preview = Pipeline(preview_config).plan()
print("Task quotas:", preview["task_quotas"])
print("Seed demonstrations:", preview["seed_ids"])

# Free local GPU check before the paid run. It checks generation, not accuracy.
probes = student_runtime.infer_many([
    {"instruction": "Reply with the JSON object {\"ok\":true}.", "input": {}},
    {"instruction": "Reply with the JSON object {\"ok\":true}.", "input": {}},
])
if any(probe.get("error") or not probe.get("generated_tokens") for probe in probes):
    raise RuntimeError("Student GPU preflight failed; inspect the worker output.")
print("Both GPU workers generated responses. No parent API calls made.")


## Run or resume

This cell spends parent API credits on training-data generation/verification and performs GPU training and evaluation. Full runs load the saved fixed validation dataset; they do not generate evaluation questions through the API and do not request human review. Completed training and saved evaluation generations are reused on resume. GitHub upload runs after each completed training round and is retriable without retraining.

The best validation checkpoint is selected and restored as `selected_checkpoint`. The independent-gold `promotion` field remains false because the benchmark is synthetic; this does not block training or GitHub uploads. Keep the same configuration/output directory to resume. Do not delete budgets or state to bypass an error.


In [ ]:
# Define automatic ZIP export before starting/resuming training.
def download_latest_adapter():
    import json
    import os
    import shutil
    from pathlib import Path
    from IPython.display import FileLink, display

    # Prevent the failed upload from blocking the next run/resume call.
    os.environ.pop("KAGGLE_ADAPTER_DATASET", None)
    BACKUP_ADAPTERS_TO_KAGGLE = False

    completed = sorted(Path(RUN_DIR).glob("round_*/training_complete.json"),
                       key=lambda p: int(p.parent.name.split("_")[-1]))
    if not completed:
        raise RuntimeError("No completed training checkpoint found in RUN_DIR. Do not delete the existing run folder.")
    checkpoint = Path(json.loads(completed[-1].read_text())["checkpoint"])
    if not (checkpoint / "adapter_config.json").is_file() or not list(checkpoint.glob("adapter_model*.safetensors")):
        raise RuntimeError(f"Adapter files are missing from {checkpoint}")
    # Only the dedicated adapter directory is archived, never API secrets or the run folder.
    download_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    archive_base = download_root / f"{Path(RUN_DIR).name}-{completed[-1].parent.name}-adapter"
    archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=checkpoint))
    print("Download:", archive, f"({archive.stat().st_size / 1024**2:.1f} MiB)")
    print("PEFT adapter + tokenizer only; load with Qwen/Qwen3-8B. This training checkpoint may not yet be evaluated.")
    print("If the link does not open, download this ZIP from Kaggle's /kaggle/working file browser.")
    display(FileLink(str(archive)))
    print("ZIP export complete. In Kaggle's file browser, open /kaggle/working and download this ZIP.")
    print("A browser download requires your click; this notebook does not attempt JavaScript auto-downloads.")


In [ ]:
# Define the GitHub uploader before training; no network requests in this cell.
"""Upload a completed local adapter to an immutable GitHub prerelease asset."""
import hashlib
import json
import os
import re
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

REPOSITORY = 'j6antony/Health-Insurance-SLM'


def request_json(method, url, token, *, payload=None, file=None, allow_missing=False):
    headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json',
               'X-GitHub-Api-Version': '2022-11-28', 'User-Agent': 'health-insurance-slm-adapter-backup'}
    data = json.dumps(payload).encode() if payload is not None else None
    if payload is not None:
        headers['Content-Type'] = 'application/json'
    if file is not None:
        headers.update({'Content-Type': 'application/zip', 'Content-Length': str(Path(file).stat().st_size)})
    try:
        if file is not None:
            with Path(file).open('rb') as stream:
                request = urllib.request.Request(url, data=stream, headers=headers, method=method)
                with urllib.request.urlopen(request, timeout=900) as response:
                    return json.load(response)
        request = urllib.request.Request(url, data=data, headers=headers, method=method)
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        status = exc.code
        exc.close()
        if allow_missing and status == 404:
            return None
        raise RuntimeError(f'GitHub {method} failed: HTTP {status}. Check repository access and Contents read/write permission. Local adapter is unchanged; rerun to resume.') from None
    except (urllib.error.URLError, TimeoutError, OSError):
        raise RuntimeError('GitHub connection failed. Local adapter is unchanged; rerun to resume.') from None


def upload_latest_adapter(run_dir, token=None, repository=REPOSITORY):
    token = token or os.environ.get('GITHUB_TOKEN', '')
    if not isinstance(token, str) or not token.strip():
        raise ValueError('Set GITHUB_TOKEN through Kaggle Secrets; never put it in source code.')
    token = token.strip()
    if not re.fullmatch(r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+', repository):
        raise ValueError('Expected owner/repository')
    run_dir = Path(run_dir)
    completed = sorted(run_dir.glob('round_*/training_complete.json'),
                       key=lambda p: int(p.parent.name.split('_')[-1]))
    if not completed:
        raise ValueError('No completed training checkpoint in RUN_DIR')
    checkpoint = Path(json.loads(completed[-1].read_text())['checkpoint'])
    if not (checkpoint/'adapter_config.json').is_file() or not list(checkpoint.glob('adapter_model*.safetensors')):
        raise ValueError('Completed adapter checkpoint is missing its configuration or weights')
    names = {'adapter_config.json', 'adapter_model.safetensors.index.json', 'tokenizer.json',
             'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt',
             'added_tokens.json', 'chat_template.jinja', 'README.md'}
    files = sorted(p for p in checkpoint.iterdir() if p.is_file() and not p.is_symlink() and
                   (p.name in names or re.fullmatch(r'adapter_model(?:-\d+-of-\d+)?\.safetensors', p.name)))
    directory = run_dir/'github_backups'; directory.mkdir(exist_ok=True)
    archive = directory/'adapter.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as output:
        for path in files:
            output.write(path, path.name)
    sha = hashlib.sha256()
    with archive.open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            sha.update(block)
    digest = sha.hexdigest()
    tag = f'adapter-{completed[-1].parent.name}-{digest[:16]}'
    api = f'https://api.github.com/repos/{repository}'
    release = request_json('GET', api+'/releases/tags/'+tag, token, allow_missing=True)
    if release is None:
        release = request_json('POST', api+'/releases', token, payload={
            'tag_name': tag, 'target_commitish': 'Distillation', 'name': tag,
            'body': 'Qwen/Qwen3-8B PEFT training checkpoint. Not a merged base model or a quality-approved release.\nZIP SHA256: '+digest,
            'draft': False, 'prerelease': True, 'make_latest': 'false'})
    assets = request_json('GET', api+f'/releases/{release["id"]}/assets?per_page=100', token)
    existing = next((asset for asset in assets if asset['name'] == 'adapter.zip'), None)
    if existing is None:
        upload_url = release['upload_url'].split('{')[0]
        parsed = urllib.parse.urlsplit(upload_url)
        if parsed.scheme != 'https' or parsed.netloc != 'uploads.github.com':
            raise ValueError('Unexpected GitHub asset upload host')
        asset = request_json('POST', upload_url+'?name=adapter.zip', token, file=archive)
    else:
        asset = existing
    if (asset.get('state') != 'uploaded' or asset.get('size') != archive.stat().st_size
            or asset.get('digest') != 'sha256:'+digest):
        raise RuntimeError('GitHub asset is not confirmed with the expected size and SHA256. No asset was overwritten. Local ZIP remains available.')
    receipt = {'release_url': release['html_url'], 'download_url': asset['browser_download_url'],
               'sha256': digest, 'checkpoint': str(checkpoint), 'status': 'github_digest_verified'}
    from self_instruct_pipeline.persistence import atomic_json
    atomic_json(directory/(tag+'.json'), receipt)
    print('Adapter uploaded and SHA256 verified:', receipt['download_url'], flush=True)
    return receipt


In [ ]:
from distillation_loop.controller import run_loop
from self_instruct_pipeline.budget import APIBudget

# Shared across development, training generation, audits, and retries.
# Counts persist across kernel restarts; rerunning does not reset the budget.
budget = APIBudget(RUN_DIR / "api_budget",
                   max_calls=120 if TEST_RUN else 2000,
                   max_cost_usd="0.50" if TEST_RUN else "5.00")
try:
    with budget.activate():
        result = run_loop(student_runtime, parent_config, loop_config)
finally:
    # Preserve completed training even if a later phase raises an error.
    import sys
    run_already_failed = sys.exc_info()[0] is not None
    if list(Path(RUN_DIR).glob("round_*/training_complete.json")):
        try:
            if UPLOAD_ADAPTERS_TO_GITHUB:
                github_upload = upload_latest_adapter(RUN_DIR)
            else:
                download_latest_adapter()
        except Exception as backup_error:
            print("Adapter backup failed:", str(backup_error))
            print("Training checkpoint is local. Retry the GitHub upload cell below; do not retrain.")
            if not run_already_failed:
                raise
student = student_runtime.model
print("Stop reason:", result["status"])
print("Completed rounds:", result["completed_rounds"])
print("Best validation checkpoint:", result["best_checkpoint"])
print("Promotion:", result.get("promotion"))
print("Release checkpoint:", result.get("release_checkpoint", "No release: synthetic smoke only"))
print("Best validation macro score:", result["best_report"]["macro_task_score"])
print("Budget accounting:", budget.state)


## Optional local ZIP fallback

GitHub is now the default automatic upload destination. This cell creates a local ZIP for manual download if needed. It does not disable GitHub uploads. Transfer the ZIP through Kaggle's file browser before ending the session.


In [ ]:
# Recreate the ZIP and display its path/link. Download it through Kaggle's file browser.
download_latest_adapter()


In [ ]:
# Inspect history without rerunning inference or training.
for entry in result["history"]:
    scores = entry["scores"]
    print({"round": entry["round"], "macro": scores["macro_task_score"],
           "minimum_task": scores["minimum_task_score"],
           "schema_evidence_valid": scores["schema_evidence_valid_rate"]})


In [ ]:
# Refresh the combined usage report, including calls from earlier sessions.
from self_instruct_pipeline.usage import export_usage
usage_summary = export_usage(RUN_DIR)
print(usage_summary)


In [ ]:
# Optional: export ALL saved Supabase history, even from previous Kaggle sessions.
from self_instruct_pipeline.usage_cloud import export_cloud_usage
if cloud_ledger:
    print(export_cloud_usage(RUN_DIR / "api_usage_all_runs.csv"))
# To export only this run:
# run_id = json.loads((RUN_DIR / "api_usage_run.json").read_text())["run_id"]
# print(export_cloud_usage(RUN_DIR / "api_usage_this_run.csv", run_id=run_id))


## Retry GitHub adapter upload

GitHub upload is enabled by default in setup and runs after each completed training round. A final retry also runs when the run/resume cell exits, including after an evaluation error if a completed checkpoint exists. The cell below retries only the upload; it does not train or evaluate. Successful uploads are SHA256-verified, and reruns reuse the matching release asset. Weights inherit the repository's visibility.


In [ ]:
# Retry GitHub upload without rerunning training or evaluation.
# The uploader is defined before the run cell; credentials load in setup.
github_upload = upload_latest_adapter(RUN_DIR)
